In [1]:
import pandas as pd
import numpy as np
import os

from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


In [5]:
PRODUCTS_PATH = r"G:\PMS_Optimization\Preprocess\Preprocessed ORANGEBOOK\products.csv"

products = pd.read_csv(
    PRODUCTS_PATH,
    dtype=str
)

print("Dataset loaded successfully.")
print("Rows    :", len(products))
print("Columns :", len(products.columns))

print("\nColumns:")
for i, col in enumerate(products.columns, 1):
    print(i, ".", col)

Dataset loaded successfully.
Rows    : 48502
Columns : 18

Columns:
1 . Ingredient
2 . DF;Route
3 . Trade_Name
4 . Applicant
5 . Strength
6 . Appl_Type
7 . Appl_No
8 . Product_No
9 . TE_Code
10 . Approval_Date
11 . RLD
12 . RS
13 . Type
14 . Applicant_Full_Name
15 . Dosage_Form
16 . Route_Of_Administration
17 . Approved_Prior_To_1982
18 . Approval_Date_ISO


In [6]:
print("TE Code distribution:")
print(
    products["TE_Code"]
    .value_counts(dropna=False)
)

TE Code distribution:
TE_Code
NaN                26649
AB                 14894
AP                  3781
AA                  1043
AB1                  588
AT                   509
AB2                  300
AB3                  129
AN                   129
AB1,AB2,AB3,AB4       72
AO                    71
AP1                   67
BX                    60
AT1                   43
AP2                   37
AB1,AB2,AB3           36
AB4                   30
AT2                   21
AB1,AB2               12
AB1,AB3               11
AT3                    9
BP                     5
AP3                    2
BD                     2
BS                     1
BC                     1
Name: count, dtype: int64


In [7]:
print("RLD distribution:")
print(
    products["RLD"]
    .value_counts(dropna=False)
)

print("\nRS distribution:")
print(
    products["RS"]
    .value_counts(dropna=False)
)

RLD distribution:
RLD
False    40716
True      7786
Name: count, dtype: int64

RS distribution:
RS
False    44019
True      4483
Name: count, dtype: int64


In [11]:
PATENT_PATH = r"G:\PMS_Optimization\Preprocess\Preprocessed ORANGEBOOK\patent.csv"

EXCLUSIVITY_PATH = (
    r"G:\PMS_Optimization\Preprocess\Preprocessed ORANGEBOOK\exclusivity.csv"
)

patent = pd.read_csv(
    PATENT_PATH,
    dtype=str
)

exclusivity = pd.read_csv(
    EXCLUSIVITY_PATH,
    dtype=str
)

print("PATENT")
print("Rows:", len(patent))
print("Columns:", len(patent.columns))

print("\nEXCLUSIVITY")
print("Rows:", len(exclusivity))
print("Columns:", len(exclusivity.columns))

PATENT
Rows: 22131
Columns: 14

EXCLUSIVITY
Rows: 2267
Columns: 6


In [12]:
key_cols = [
    "Appl_Type",
    "Appl_No",
    "Product_No"
]

for dataframe in [
    products,
    patent,
    exclusivity
]:

    for col in key_cols:

        dataframe[col] = (
            dataframe[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

print("Key columns cleaned.")

Key columns cleaned.


In [13]:
products["RLD"] = (
    products["RLD"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
    .fillna(False)
)

products["RS"] = (
    products["RS"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
    .fillna(False)
)

print("RLD:")
print(products["RLD"].value_counts())

print("\nRS:")
print(products["RS"].value_counts())

RLD:
RLD
False    40716
True      7786
Name: count, dtype: int64

RS:
RS
False    44019
True      4483
Name: count, dtype: int64


In [14]:
group_cols = [
    "Ingredient",
    "Strength",
    "Dosage_Form",
    "Route_Of_Administration"
]

rld_groups = (
    products
    .groupby(group_cols)["RLD"]
    .any()
    .reset_index(name="Has_RLD")
)

rld_groups = rld_groups[
    rld_groups["Has_RLD"] == True
].copy()

print(
    "Number of groups containing RLD:",
    len(rld_groups)
)

Number of groups containing RLD: 7109


In [15]:
products_with_rld = products.merge(
    rld_groups[group_cols],
    on=group_cols,
    how="inner"
)

print(
    "Products in RLD groups:",
    len(products_with_rld)
)

rld_products = products_with_rld[
    products_with_rld["RLD"] == True
].copy()

alternatives = products_with_rld[
    products_with_rld["RLD"] == False
].copy()

print("RLD products:", len(rld_products))
print("Non-RLD candidates:", len(alternatives))

Products in RLD groups: 29945
RLD products: 7772
Non-RLD candidates: 22173


In [16]:
alternatives["TE_Code"] = (
    alternatives["TE_Code"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

alternatives["AB_Family"] = (
    alternatives["TE_Code"]
    .str.startswith("AB")
)

pbm_candidates = alternatives[
    alternatives["AB_Family"] == True
].copy()

print(
    "Total candidate alternatives:",
    len(alternatives)
)

print(
    "AB-family candidates:",
    len(pbm_candidates)
)

display(
    pbm_candidates[
        [
            "Ingredient",
            "Strength",
            "Dosage_Form",
            "Route_Of_Administration",
            "Trade_Name",
            "TE_Code",
            "RLD",
            "RS"
        ]
    ].head(20)
)

Total candidate alternatives: 22173
AB-family candidates: 9919


,Ingredient,Strength,Dosage_Form,Route_Of_Administration,Trade_Name,TE_Code,RLD,RS
0,BUDESONIDE,2MG/ACTUATION,"AEROSOL, FOAM",RECTAL,BUDESONIDE,AB,False,True
33,ALBUTEROL SULFATE,EQ 0.09MG BASE/INH,"AEROSOL, METERED",INHALATION,ALBUTEROL SULFATE,AB2,False,False
34,ALBUTEROL SULFATE,EQ 0.09MG BASE/INH,"AEROSOL, METERED",INHALATION,ALBUTEROL SULFATE,AB2,False,False
35,ALBUTEROL SULFATE,EQ 0.09MG BASE/INH,"AEROSOL, METERED",INHALATION,ALBUTEROL SULFATE,AB1,False,False
36,ALBUTEROL SULFATE,EQ 0.09MG BASE/INH,"AEROSOL, METERED",INHALATION,ALBUTEROL SULFATE,AB3,False,False
37,ALBUTEROL SULFATE,EQ 0.09MG BASE/INH,"AEROSOL, METERED",INHALATION,ALBUTEROL SULFATE,AB2,False,False
38,ALBUTEROL SULFATE,EQ 0.09MG BASE/INH,"AEROSOL, METERED",INHALATION,ALBUTEROL SULFATE,AB2,False,False
39,ALBUTEROL SULFATE,EQ 0.09MG BASE/INH,"AEROSOL, METERED",INHALATION,ALBUTEROL SULFATE,AB1,False,False
49,IPRATROPIUM BROMIDE,0.021MG/INH,"AEROSOL, METERED",INHALATION,IPRATROPIUM BROMIDE,AB,False,False
55,BUDESONIDE; FORMOTEROL FUMARATE DIHYDRATE,0.08MG/INH;0.0045MG/INH,"AEROSOL, METERED",INHALATION,BREYNA,AB,False,False


In [17]:
rld_lookup = (
    rld_products[
        group_cols +
        [
            "Trade_Name",
            "Appl_No",
            "Product_No",
            "TE_Code"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "Trade_Name": "RLD_Trade_Name",
            "Appl_No": "RLD_Appl_No",
            "Product_No": "RLD_Product_No",
            "TE_Code": "RLD_TE_Code"
        }
    )
)

print("RLD lookup records:", len(rld_lookup))

RLD lookup records: 7772


In [18]:
pbm_candidates = pbm_candidates.merge(
    rld_lookup,
    on=group_cols,
    how="left"
)

print(
    "Candidate records:",
    len(pbm_candidates)
)

print(
    "Mapped to RLD:",
    pbm_candidates["RLD_Trade_Name"]
    .notna()
    .sum()
)

print(
    "Without RLD mapping:",
    pbm_candidates["RLD_Trade_Name"]
    .isna()
    .sum()
)

Candidate records: 10969
Mapped to RLD: 10969
Without RLD mapping: 0


In [19]:
patent_summary = (
    patent
    .groupby(
        key_cols,
        as_index=False
    )
    .agg(
        Patent_Count=(
            "Patent_No",
            "nunique"
        ),

        Patent_Numbers=(
            "Patent_No",
            lambda x:
            ", ".join(
                x.dropna()
                .astype(str)
                .unique()
            )
        ),

        Patent_Expiry=(
            "Patent_Expire_Date_ISO",
            lambda x:
            ", ".join(
                x.dropna()
                .astype(str)
                .unique()
            )
        ),

        Patent_Use_Codes=(
            "Patent_Use_Code",
            lambda x:
            ", ".join(
                x.dropna()
                .astype(str)
                .unique()
            )
        )
    )
)

print(
    "Patent summary:",
    patent_summary.shape
)

Patent summary: (2634, 7)


In [20]:
exclusivity_summary = (
    exclusivity
    .groupby(
        key_cols,
        as_index=False
    )
    .agg(
        Exclusivity_Count=(
            "Exclusivity_Code",
            "nunique"
        ),

        Exclusivity_Codes=(
            "Exclusivity_Code",
            lambda x:
            ", ".join(
                x.dropna()
                .astype(str)
                .unique()
            )
        ),

        Exclusivity_Dates=(
            "Exclusivity_Date_ISO",
            lambda x:
            ", ".join(
                x.dropna()
                .astype(str)
                .unique()
            )
        )
    )
)

print(
    "Exclusivity summary:",
    exclusivity_summary.shape
)

Exclusivity summary: (1192, 6)


In [21]:
pbm_candidates = pbm_candidates.merge(
    patent_summary,
    left_on=[
        "Appl_Type",
        "Appl_No",
        "Product_No"
    ],
    right_on=key_cols,
    how="left"
)

# Remove duplicate key columns created by merge
for col in [
    "Appl_Type_y",
    "Appl_No_y",
    "Product_No_y"
]:

    if col in pbm_candidates.columns:
        pbm_candidates.drop(
            columns=[col],
            inplace=True
        )

print(
    "Patent information attached."
)

Patent information attached.


In [22]:
pbm_candidates = pbm_candidates.merge(
    exclusivity_summary,
    left_on=[
        "Appl_Type",
        "Appl_No",
        "Product_No"
    ],
    right_on=key_cols,
    how="left"
)

for col in [
    "Appl_Type_y",
    "Appl_No_y",
    "Product_No_y"
]:

    if col in pbm_candidates.columns:
        pbm_candidates.drop(
            columns=[col],
            inplace=True
        )

print(
    "Exclusivity information attached."
)

Exclusivity information attached.


In [23]:
pbm_candidates["Patent_Count"] = (
    pd.to_numeric(
        pbm_candidates["Patent_Count"],
        errors="coerce"
    )
    .fillna(0)
)

pbm_candidates["Exclusivity_Count"] = (
    pd.to_numeric(
        pbm_candidates["Exclusivity_Count"],
        errors="coerce"
    )
    .fillna(0)
)

print(
    pbm_candidates[
        [
            "Trade_Name",
            "RLD_Trade_Name",
            "TE_Code",
            "Patent_Count",
            "Exclusivity_Count"
        ]
    ].head(20)
)

           Trade_Name RLD_Trade_Name TE_Code  Patent_Count  Exclusivity_Count
0          BUDESONIDE         UCERIS      AB           0.0                0.0
1   ALBUTEROL SULFATE     PROAIR HFA     AB2           0.0                0.0
2   ALBUTEROL SULFATE  PROVENTIL-HFA     AB2           0.0                0.0
3   ALBUTEROL SULFATE   VENTOLIN HFA     AB2           0.0                0.0
4   ALBUTEROL SULFATE     PROAIR HFA     AB2           0.0                0.0
5   ALBUTEROL SULFATE  PROVENTIL-HFA     AB2           0.0                0.0
6   ALBUTEROL SULFATE   VENTOLIN HFA     AB2           0.0                0.0
7   ALBUTEROL SULFATE     PROAIR HFA     AB1           0.0                0.0
8   ALBUTEROL SULFATE  PROVENTIL-HFA     AB1           0.0                0.0
9   ALBUTEROL SULFATE   VENTOLIN HFA     AB1           0.0                0.0
10  ALBUTEROL SULFATE     PROAIR HFA     AB3           0.0                1.0
11  ALBUTEROL SULFATE  PROVENTIL-HFA     AB3           0.0      

In [24]:
print("=" * 60)
print("CANDIDATE GENERATION SUMMARY")
print("=" * 60)

print(
    "Total Orange Book products:",
    len(products)
)

print(
    "RLD products:",
    len(rld_products)
)

print(
    "AB-family candidates:",
    len(pbm_candidates)
)

print(
    "Candidates with RLD mapping:",
    pbm_candidates[
        "RLD_Trade_Name"
    ].notna().sum()
)

print(
    "Candidates without RLD mapping:",
    pbm_candidates[
        "RLD_Trade_Name"
    ].isna().sum()
)

CANDIDATE GENERATION SUMMARY
Total Orange Book products: 48502
RLD products: 7772
AB-family candidates: 10969
Candidates with RLD mapping: 10969
Candidates without RLD mapping: 0


In [25]:
OUTPUT_DIR = r"outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

candidate_path = os.path.join(
    OUTPUT_DIR,
    "PBM_Candidate_Alternatives.csv"
)

pbm_candidates.to_csv(
    candidate_path,
    index=False
)

print(
    "Candidate dataset saved:"
)

print(candidate_path)

Candidate dataset saved:
outputs\PBM_Candidate_Alternatives.csv
